
# Interactive Lyot filter simulation

This notebook is a Python/Jupyter version of the MATLAB Lyot-filter simulations.

The main ideas are:

1. A **polarizer** projects the electric field onto one linear polarization direction.
2. A **retarder / waveplate** introduces a wavelength-dependent phase delay between the two field components.
3. A **Lyot stage** converts this wavelength-dependent phase into wavelength-dependent transmission by placing a retarder between polarizers.
4. Multiple stages sharpen the transmission peaks when the retardances are chosen in a simple ratio, for example `600 nm`, `300 nm`, `150 nm`.

Run the cells from top to bottom. The interactive sliders require `ipywidgets`.



## 1. Mathematical model

We describe polarized light by a Jones vector

$$
\mathbf E = \begin{pmatrix} E_x \\ E_y \end{pmatrix}.
$$

A linear polarizer at angle $\alpha$ is represented by

$$
P(\alpha) =
\begin{pmatrix}
\cos^2\alpha & \sin\alpha\cos\alpha \\
\sin\alpha\cos\alpha & \sin^2\alpha
\end{pmatrix}.
$$

A retarder aligned with the $x/y$ axes is represented by

$$
R(\Deltaϕ
\begin{pmatrix}
\exp(i\Deltaϕ) & 0 \\
0 & 1
\end{pmatrix}.
$$

The phase retardance is wavelength dependent. If the optical path delay is given as $\Lambda=d\cdot|n_\perp-n_∥|$, then

$$
\Delta\phi(\lambda) = 2\pi\,\frac{\Lambda}{\lambda}.
$$

This is the key reason why the transmission of a Lyot filter depends on wavelength.


In [15]:
# @title

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

plt.rcParams.update({
    "figure.figsize": (9, 4.8),
    "axes.grid": True,
    "font.size": 11,
})


In [16]:
# @title

def normalize(E):
    """Normalize a Jones vector unless its norm is zero."""
    E = np.asarray(E, dtype=complex).reshape(2)
    n = np.linalg.norm(E)
    return E if n == 0 else E / n


def jones_polarizer(alpha_deg):
    """Jones matrix of a linear polarizer at angle alpha_deg."""
    a = np.deg2rad(alpha_deg)
    c, s = np.cos(a), np.sin(a)
    return np.array([[c*c, s*c], [s*c, s*s]], dtype=complex)


def jones_retarder(retardance_deg):
    """Jones matrix of an x/y-aligned retarder. The x component is delayed."""
    d = np.deg2rad(retardance_deg)
    return np.array([[np.exp(1j*d), 0], [0, 1]], dtype=complex)


def retardance_deg_from_opd(opd_nm, wavelength_nm, tilt_factor=1.0):
    """Convert optical path delay in nm to phase retardance in degrees."""
    return tilt_factor * 360.0 * opd_nm / wavelength_nm


def input_jones(kind="linear x", linear_angle_deg=0.0, ellipticity_phase_deg=90.0):
    """Convenience function for common input polarization states."""
    if kind == "linear x":
        return np.array([1, 0], dtype=complex)
    if kind == "linear y":
        return np.array([0, 1], dtype=complex)
    if kind == "linear angle":
        a = np.deg2rad(linear_angle_deg)
        return normalize([np.cos(a), np.sin(a)])
    if kind == "circular":
        return normalize([1, np.exp(1j*np.deg2rad(ellipticity_phase_deg))])
    raise ValueError(f"Unknown input kind: {kind}")


def apply_filter(wavelength_nm, E0, pol_angles, opd_nms, active_pols, active_rets, tilt_factor=1.0):
    """
    Apply a cascade P1 - R1 - P2 - R2 - P3 - R3 - P4.
    Returns final field and intermediate fields after each completed stage.
    """
    E = normalize(E0)
    I = np.eye(2, dtype=complex)

    P = [jones_polarizer(a) if on else I for a, on in zip(pol_angles, active_pols)]
    R = []
    for opd, on in zip(opd_nms, active_rets):
        delta = retardance_deg_from_opd(opd, wavelength_nm, tilt_factor)
        R.append(jones_retarder(delta) if on else I)

    E = P[0] @ E
    after = {"after P1": E}

    E = R[0] @ E
    after["after R1"] = E
    E = P[1] @ E
    after["stage 1"] = E

    E = R[1] @ E
    after["after R2"] = E
    E = P[2] @ E
    after["stage 2"] = E

    E = R[2] @ E
    after["after R3"] = E
    E = P[3] @ E
    after["stage 3"] = E

    return E, after


def spectrum(wavelengths_nm, E0, pol_angles, opd_nms, active_pols, active_rets, tilt_factor=1.0):
    """Compute total and per-stage amplitude/intensity spectra."""
    amp_all = []
    amp_stage = [[], [], []]

    for lam in wavelengths_nm:
        E_final, after = apply_filter(lam, E0, pol_angles, opd_nms, active_pols, active_rets, tilt_factor)
        amp_all.append(np.linalg.norm(E_final))
        for k, name in enumerate(["stage 1", "stage 2", "stage 3"]):
            amp_stage[k].append(np.linalg.norm(after[name]))

    amp_all = np.asarray(amp_all)
    amp_stage = [np.asarray(a) for a in amp_stage]
    return amp_all, amp_stage


def polarization_trace(E, n=240):
    """Return x(t), y(t) over one optical cycle for visualizing the polarization ellipse."""
    t = np.linspace(0, 2*np.pi, n)
    E = np.asarray(E, dtype=complex).reshape(2)
    x = np.real(E[0] * np.exp(-1j*t))
    y = np.real(E[1] * np.exp(-1j*t))
    return x, y



## 2. Interactive transmission spectrum

Choose polarizer angles, optical path delays, active/inactive elements, and whether to plot electric-field amplitude or intensity.

Useful things to try:

- Set the stages to `600`, `300`, `150 nm`. This is the standard Lyot idea: each successive stage has half the retardance.
- Disable the later stages and compare the width of the transmission peaks.
- Change all polarizers from `45°` to `0°` or `90°` and see why the filter stops working.
- Move the “tilt factor”. In a real birefringent plate, tilting changes the effective optical path difference, so the passband shifts.


In [17]:
# @title

def plot_interactive_spectrum(
    input_kind="linear x",
    input_angle=0.0,
    circular_phase=90.0,
    lambda_min=350,
    lambda_max=1200,
    plot_quantity="intensity",
    tilt_factor=1.0,
    pol1_on=True, pol1_angle=45.0,
    ret1_on=True, ret1_opd=500.0,
    pol2_on=True, pol2_angle=45.0,
    ret2_on=False, ret2_opd=300.0,
    pol3_on=False, pol3_angle=45.0,
    ret3_on=False, ret3_opd=1000.0,
    pol4_on=False, pol4_angle=45.0,
):
    wavelengths = np.linspace(lambda_min, lambda_max, 900)
    E0 = input_jones(input_kind, input_angle, circular_phase)
    pol_angles = [pol1_angle, pol2_angle, pol3_angle, pol4_angle]
    opd_nms = [ret1_opd, ret2_opd, ret3_opd]
    active_pols = [pol1_on, pol2_on, pol3_on, pol4_on]
    active_rets = [ret1_on, ret2_on, ret3_on]

    amp_all, amp_stage = spectrum(wavelengths, E0, pol_angles, opd_nms, active_pols, active_rets, tilt_factor)

    if plot_quantity == "intensity":
        y_all = amp_all**2
        y_stage = [a**2 for a in amp_stage]
        ylabel = "relative intensity"
    else:
        y_all = amp_all
        y_stage = amp_stage
        ylabel = "relative electric-field amplitude"

    fig, ax = plt.subplots(figsize=(9.5, 5.2))
    ax.plot(wavelengths, y_all, linewidth=2.5, label="all active stages")
    ax.plot(wavelengths, y_stage[0], ":", linewidth=2, label="after stage 1")
    ax.plot(wavelengths, y_stage[1], ":", linewidth=2, label="after stage 2")
    ax.plot(wavelengths, y_stage[2], ":", linewidth=2, label="after stage 3")
    ax.set_xlim(lambda_min, lambda_max)
    ax.set_ylim(-0.03, 1.03)
    ax.set_xlabel("wavelength λ [nm]")
    ax.set_ylabel(ylabel)
    ax.set_title("Lyot filter transmission")
    ax.legend(loc="upper right")
    plt.show()

    # Small numerical readout near the strongest passband.
    peak_idx = np.argmax(y_all)
    print(f"Peak in shown range: λ ≈ {wavelengths[peak_idx]:.1f} nm, transmission ≈ {y_all[peak_idx]:.3f}")

style = {"description_width": "120px"}
layout = widgets.Layout(width="330px")

controls_left = widgets.VBox([
    widgets.Dropdown(options=["linear x", "linear y", "linear angle", "circular"], value="linear x", description="input", style=style, layout=layout),
    widgets.FloatSlider(value=0, min=-90, max=90, step=1, description="input angle [°]", style=style, layout=layout),
    widgets.FloatSlider(value=90, min=-180, max=180, step=5, description="circular phase [°]", style=style, layout=layout),
    widgets.IntRangeSlider(value=[350, 900], min=100, max=1200, step=10, description="λ range [nm]", style=style, layout=layout),
    widgets.Dropdown(options=["intensity", "amplitude"], value="intensity", description="plot", style=style, layout=layout),
    widgets.FloatSlider(value=1.0, min=0.5, max=1.5, step=0.01, description="tilt factor", style=style, layout=layout),
])

controls_stage1 = widgets.VBox([
    widgets.HTML("<b>Stage 1</b>"),
    widgets.Checkbox(value=True, description="polarizer 1 on"),
    widgets.FloatSlider(value=45, min=-90, max=90, step=1, description="pol. 1 [°]", style=style, layout=layout),
    widgets.Checkbox(value=True, description="retarder 1 on"),
    widgets.FloatSlider(value=500, min=0, max=2500, step=10, description="OPD 1 [nm]", style=style, layout=layout),
])

controls_stage2 = widgets.VBox([
    widgets.HTML("<b>Stage 2</b>"),
    widgets.Checkbox(value=True, description="polarizer 2 on"),
    widgets.FloatSlider(value=45, min=-90, max=90, step=1, description="pol. 2 [°]", style=style, layout=layout),
    widgets.Checkbox(value=False, description="retarder 2 on"),
    widgets.FloatSlider(value=1000, min=0, max=2500, step=10, description="OPD 2 [nm]", style=style, layout=layout),
    widgets.Checkbox(value=False, description="polarizer 3 on"),
    widgets.FloatSlider(value=45, min=-90, max=90, step=1, description="pol. 3 [°]", style=style, layout=layout),
])

controls_stage3 = widgets.VBox([
    widgets.HTML("<b>Optional stage 3</b>"),
    widgets.Checkbox(value=False, description="retarder 3 on"),
    widgets.FloatSlider(value=150, min=0, max=1200, step=10, description="OPD 3 [nm]", style=style, layout=layout),
    widgets.Checkbox(value=False, description="polarizer 4 on"),
    widgets.FloatSlider(value=45, min=-90, max=90, step=1, description="pol. 4 [°]", style=style, layout=layout),
])

# Link widget values to function arguments.
ui = widgets.VBox([
    widgets.HBox([controls_left, controls_stage1]),
    widgets.HBox([controls_stage2, controls_stage3]),
])

out = widgets.interactive_output(plot_interactive_spectrum, {
    "input_kind": controls_left.children[0],
    "input_angle": controls_left.children[1],
    "circular_phase": controls_left.children[2],
    "lambda_min": widgets.fixed(350),  # overwritten below by wrapper
    "lambda_max": widgets.fixed(900),
    "plot_quantity": controls_left.children[4],
    "tilt_factor": controls_left.children[5],
    "pol1_on": controls_stage1.children[1],
    "pol1_angle": controls_stage1.children[2],
    "ret1_on": controls_stage1.children[3],
    "ret1_opd": controls_stage1.children[4],
    "pol2_on": controls_stage2.children[1],
    "pol2_angle": controls_stage2.children[2],
    "ret2_on": controls_stage2.children[3],
    "ret2_opd": controls_stage2.children[4],
    "pol3_on": controls_stage2.children[5],
    "pol3_angle": controls_stage2.children[6],
    "ret3_on": controls_stage3.children[1],
    "ret3_opd": controls_stage3.children[2],
    "pol4_on": controls_stage3.children[3],
    "pol4_angle": controls_stage3.children[4],
})

# interactive_output cannot directly unpack an IntRangeSlider into two arguments,
# so we use a compact wrapper instead.
def plot_interactive_spectrum_with_range(lambda_range, **kwargs):
    return plot_interactive_spectrum(lambda_min=lambda_range[0], lambda_max=lambda_range[1], **kwargs)

out = widgets.interactive_output(plot_interactive_spectrum_with_range, {
    "input_kind": controls_left.children[0],
    "input_angle": controls_left.children[1],
    "circular_phase": controls_left.children[2],
    "lambda_range": controls_left.children[3],
    "plot_quantity": controls_left.children[4],
    "tilt_factor": controls_left.children[5],
    "pol1_on": controls_stage1.children[1],
    "pol1_angle": controls_stage1.children[2],
    "ret1_on": controls_stage1.children[3],
    "ret1_opd": controls_stage1.children[4],
    "pol2_on": controls_stage2.children[1],
    "pol2_angle": controls_stage2.children[2],
    "ret2_on": controls_stage2.children[3],
    "ret2_opd": controls_stage2.children[4],
    "pol3_on": controls_stage2.children[5],
    "pol3_angle": controls_stage2.children[6],
    "ret3_on": controls_stage3.children[1],
    "ret3_opd": controls_stage3.children[2],
    "pol4_on": controls_stage3.children[3],
    "pol4_angle": controls_stage3.children[4],
})

display(ui, out)


Output()


## 3. Polarization-state view

The previous plot shows the final spectral effect. This view shows **what happens to the electric field** after each element for one selected wavelength.

A polarizer turns the field into a line. A retarder generally turns a linearly polarized field into an ellipse, because the two components acquire a relative phase. The next polarizer then converts this changed polarization state into a different transmitted amplitude.

This is a good way to explain why a Lyot filter works: the retarder does not directly “absorb” wavelengths. It changes polarization in a wavelength-dependent way, and the following polarizer converts that polarization change into wavelength-dependent intensity.


In [18]:
# @title

def plot_polarization_states(
    wavelength_nm=550.0,
    input_kind="circular",
    input_angle=0.0,
    circular_phase=90.0,
    tilt_factor=1.0,
    ret1_opd=600.0,
    ret2_opd=300.0,
    pol_angle=45.0,
):
    E0 = input_jones(input_kind, input_angle, circular_phase)
    P = jones_polarizer(pol_angle)
    R1 = jones_retarder(retardance_deg_from_opd(ret1_opd, wavelength_nm, tilt_factor))
    R2 = jones_retarder(retardance_deg_from_opd(ret2_opd, wavelength_nm, tilt_factor))

    states = []
    names = []

    E = normalize(E0)
    states.append(E); names.append("input")
    E = P @ E
    states.append(E); names.append("after polarizer 1")
    E = R1 @ E
    states.append(E); names.append("after retarder 1")
    E = P @ E
    states.append(E); names.append("after polarizer 2")
    E = R2 @ E
    states.append(E); names.append("after retarder 2")
    E = P @ E
    states.append(E); names.append("after polarizer 3")

    fig, axes = plt.subplots(2, 3, figsize=(10, 6), constrained_layout=True)
    axes = axes.ravel()
    max_amp = max(np.linalg.norm(s) for s in states)
    lim = max(1e-12, max_amp) * 1.05

    for ax, E, name in zip(axes, states, names):
        x, y = polarization_trace(E)
        ax.plot(x, y, linewidth=2)
        ax.plot([0, np.real(E[0])], [0, np.real(E[1])], "o-", alpha=0.5)
        ax.set_aspect("equal", adjustable="box")
        ax.set_xlim(-lim, lim)
        ax.set_ylim(-lim, lim)
        ax.set_xlabel("$E_x$")
        ax.set_ylabel("$E_y$")
        ax.set_title(f"{name}\nI = {np.linalg.norm(E)**2:.3f}")

    fig.suptitle(f"Polarization evolution at λ = {wavelength_nm:.0f} nm", y=1.02)
    plt.show()

    delta1 = retardance_deg_from_opd(ret1_opd, wavelength_nm, tilt_factor)
    delta2 = retardance_deg_from_opd(ret2_opd, wavelength_nm, tilt_factor)
    print(f"Retardance at this wavelength: δ1 = {delta1:.1f}°, δ2 = {delta2:.1f}°")

widgets.interact(
    plot_polarization_states,
    wavelength_nm=widgets.FloatSlider(value=550, min=350, max=900, step=5, description="λ [nm]", style=style, layout=layout),
    input_kind=widgets.Dropdown(options=["linear x", "linear y", "linear angle", "circular"], value="circular", description="input", style=style, layout=layout),
    input_angle=widgets.FloatSlider(value=0, min=-90, max=90, step=1, description="input angle [°]", style=style, layout=layout),
    circular_phase=widgets.FloatSlider(value=90, min=-180, max=180, step=5, description="circular phase [°]", style=style, layout=layout),
    tilt_factor=widgets.FloatSlider(value=1.0, min=0.5, max=1.5, step=0.01, description="tilt factor", style=style, layout=layout),
    ret1_opd=widgets.FloatSlider(value=600, min=0, max=1200, step=10, description="OPD 1 [nm]", style=style, layout=layout),
    ret2_opd=widgets.FloatSlider(value=300, min=0, max=1200, step=10, description="OPD 2 [nm]", style=style, layout=layout),
    pol_angle=widgets.FloatSlider(value=45, min=-90, max=90, step=1, description="polarizers [°]", style=style, layout=layout),
);


interactive(children=(FloatSlider(value=550.0, description='λ [nm]', layout=Layout(width='330px'), max=900.0, …